<a href="https://colab.research.google.com/github/mochamadx5/MachineLearning_11_MochamadReza/blob/main/JS06/JS06_TugasLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [9]:
# Load data
df = pd.read_csv("voice.csv")

In [10]:
# Preprocess
X = df.drop('label', axis=1)
y = df['label']

# Encode label
le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = []
splits = {'70:30': 0.3, '80:20': 0.2}
kernels = ['linear', 'poly', 'rbf']

In [12]:
for split_name, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=test_size, random_state=42)
    for kernel in kernels:
        model = SVC(kernel=kernel, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append({
            'Split': split_name,
            'Kernel': kernel.capitalize(),
            'Akurasi (%)': round(acc * 100, 2)
        })

results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

| Split   | Kernel   |   Akurasi (%) |
|:--------|:---------|--------------:|
| 70:30   | Linear   |         97.06 |
| 70:30   | Poly     |         95.69 |
| 70:30   | Rbf      |         98.11 |
| 80:20   | Linear   |         97.63 |
| 80:20   | Poly     |         96.85 |
| 80:20   | Rbf      |         98.26 |


In [13]:
import zipfile
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Ekstrak dataset gambar
zip_path = '/content/images.zip'
extract_path = '/content/extracted_images'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Ekstraksi selesai. Isi folder:", os.listdir(extract_path))

Ekstraksi selesai. Isi folder: ['__MACOSX', 'images']


In [16]:
# 2. Fungsi untuk ekstraksi fitur histogram warna RGB
def extract_color_histogram(image_path, bins=(8, 8, 8)):
    image = cv2.imread(image_path)
    if image is None:
        return None
    # Konversi BGR bawaan OpenCV ke RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    # Hitung histogram 3D untuk saluran R, G, B dan normalisasi
    hist = cv2.calcHist([image], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

# Load data gambar dan label
X_img = []
y_img = []

# Definisikan kategori berdasarkan struktur folder di dalam zip (siang/day dan malam/night)
for root, dirs, files in os.walk(extract_path):
    # Abaikan direktori metadata __MACOSX
    if '__MACOSX' in root:
        continue
    for file in files:
        # Abaikan file tersembunyi/metadata macOS
        if file.startswith('._'):
            continue
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(root, file)

            # Menentukan label dari nama folder atau nama file
            rel_path = os.path.relpath(file_path, extract_path).lower()
            if 'day' in rel_path or 'siang' in rel_path:
                label = 1  # Siang
            elif 'night' in rel_path or 'malam' in rel_path:
                label = 0  # Malam
            else:
                continue

            features = extract_color_histogram(file_path)
            if features is not None:
                X_img.append(features)
                y_img.append(label)

X_img = np.array(X_img)
y_img = np.array(y_img)

print(f"Total data gambar: {len(X_img)}")
print(f"Distribusi kelas (1 = Siang, 0 = Malam): {np.bincount(y_img)}")

Total data gambar: 400
Distribusi kelas (1 = Siang, 0 = Malam): [200 200]


In [17]:
# 3. Split data (80:20) & Standarisasi
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, y_img, test_size=0.2, random_state=42, stratify=y_img
)

scaler_img = StandardScaler()
X_train_scaled = scaler_img.fit_transform(X_train_img)
X_test_scaled = scaler_img.transform(X_test_img)

# 4. Hyperparameter Tuning SVM RBF menggunakan GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

grid_search = GridSearchCV(SVC(kernel='rbf', random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train_img)

# Evaluasi Model Terbaik
best_model = grid_search.best_estimator_
y_pred_img = best_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test_img, y_pred_img)

print("Hyperparameter Terbaik:", grid_search.best_params_)
print(f"Akurasi Model Siang-Malam: {accuracy * 100:.2f}%")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test_img, y_pred_img, target_names=['Malam (Night)', 'Siang (Day)']))

Hyperparameter Terbaik: {'C': 100, 'gamma': 0.001}
Akurasi Model Siang-Malam: 97.50%

Laporan Klasifikasi:
               precision    recall  f1-score   support

Malam (Night)       0.95      1.00      0.98        40
  Siang (Day)       1.00      0.95      0.97        40

     accuracy                           0.97        80
    macro avg       0.98      0.97      0.97        80
 weighted avg       0.98      0.97      0.97        80

